# Data cleaning and merge

## 0. Setup

In [2]:
import csv
from pathlib import Path

import pandas as pd

raw_data_path = Path.cwd().parent / "data" / "raw"
raw_files = list(raw_data_path.glob("*.csv"))

print(f"Found {len(raw_files)} raw data files:")
for file in raw_files:
    print(f"- {file.name}")

Found 3 raw data files:
- (2.2.7_005)Rendiconto_Entrate.csv
- (2.2.7_008)Rendiconto_Spese_Riepilogo_Missioni.csv
- (2.2.7_010)Rendiconto_Quadro_Generale_Riassuntivo.csv


## 1. First inspection

### 1.1 Mission Expenses Overview
Open "(2.2.7_008)Rendiconto_Spese_Riepilogo_Missioni.csv", inspect columns, datatypes and ecoding. Find out how Comune column is called and which value has for "Cagliari"

In [3]:
mission_exp_overview = pd.read_csv(raw_data_path / raw_files[1], nrows=100)

print(f"\nShape: {mission_exp_overview.shape[0]} rows, {mission_exp_overview.shape[1]} columns")
print(f"Columns: {mission_exp_overview.columns.tolist()}")
print(f"Data types: {mission_exp_overview.dtypes.to_dict()}")
print(mission_exp_overview.head())
print(f"Missing values: {mission_exp_overview.isnull().sum().to_dict()}")
print(f"Encoding: {mission_exp_overview.encoding if hasattr(mission_exp_overview, 'encoding') else 'unknown'}")


Shape: 100 rows, 12 columns
Columns: ['Codice Voce Riepilogo Spese', 'Esercizio Finanziario', 'Ordine Esposizione Riepilogo Missioni Voce di Riepilogo Ente', 'Descrizione Zona', 'Territorio Regione', 'Territorio Provincia', 'Territorio Comune', 'Descrizione Ente', 'Descrizione Tipologia Ente', 'Tipo Soggetto', 'Previsioni Definitive in CC', 'Descrizione Voce Riepilogo Spese']
Data types: {'Codice Voce Riepilogo Spese': dtype('int64'), 'Esercizio Finanziario': dtype('int64'), 'Ordine Esposizione Riepilogo Missioni Voce di Riepilogo Ente': dtype('int64'), 'Descrizione Zona': <StringDtype(storage='python', na_value=nan)>, 'Territorio Regione': <StringDtype(storage='python', na_value=nan)>, 'Territorio Provincia': <StringDtype(storage='python', na_value=nan)>, 'Territorio Comune': <StringDtype(storage='python', na_value=nan)>, 'Descrizione Ente': <StringDtype(storage='python', na_value=nan)>, 'Descrizione Tipologia Ente': <StringDtype(storage='python', na_value=nan)>, 'Tipo Soggetto': <St

## 2. Chunk filtering

In [5]:
def extract_filtered_tables(
    file_path: Path, comune_target: str = "CAGLIARI", separator: str = ","
) -> dict[str, pd.DataFrame]:
    """Reads a multi-table CSV file row by row, filtering for a specific comune and returning a dictionary of different DataFrames whenever intestation changes."""

    final_tables = {}
    tables_data = {}

    cum_rows = []
    current_header = None
    row_length = 0

    comune_index = None

    with open(file_path, mode="r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter=separator)

        for row in reader:
            if row_length != len(row):
                row_length = len(row)
                if current_header is not None:
                    tables_data[current_header] = cum_rows
                current_header = tuple(row)
                cum_rows = []

                try:
                    comune_index = current_header.index("Territorio Comune")
                except ValueError:
                    comune_index = None

                print(row)
            else:
                if comune_index is not None:
                    if row[comune_index] == comune_target:
                        cum_rows.append(row)
                else:
                    cum_rows.append(row)

        if current_header is not None:
            tables_data[current_header] = cum_rows

        for header, rows in tables_data.items():
            final_tables[header] = pd.DataFrame(rows, columns=header)

        return final_tables

In [19]:
exp_overview_mission_dfs = extract_filtered_tables(raw_data_path / raw_files[1])

['Codice Voce Riepilogo Spese', 'Esercizio Finanziario', 'Ordine Esposizione Riepilogo Missioni Voce di Riepilogo Ente', 'Descrizione Zona', 'Territorio Regione', 'Territorio Provincia', 'Territorio Comune', 'Descrizione Ente', 'Descrizione Tipologia Ente', 'Tipo Soggetto', 'Previsioni Definitive in CC', 'Descrizione Voce Riepilogo Spese']
['Descrizione Zona', 'Descrizione Tipologia Ente', 'Tipo Soggetto', 'Esercizio Finanziario', 'Territorio Regione', 'Territorio Provincia', 'Territorio Comune', 'Descrizione Ente', 'Codice Missione Arconet', 'Descrizione Missione Arconet', 'Residui Passivi Iniziali', 'Pagamenti in CR', 'Riaccertamenti Residui', 'Residui Passivi in CR', 'Previsioni Definitive in CC', 'Pagamenti in CC', 'Impegni', 'Economie in CC', 'Residui Passivi in CC', 'Previsioni Definitive di Cassa', 'Totale Pagamenti', 'Fondo Pluriennale Vincolato', 'Residui Passivi da riportare totale']
['Data Osservazione']
['Codice Voce Riepilogo Spese', 'Anno VRS', 'Descrizione Voce Riepilogo

In [22]:
for df in exp_overview_mission_dfs.values():
    if 'Descrizione Tipologia Ente' in df.columns:
        print(df['Descrizione Tipologia Ente'].value_counts())

Descrizione Tipologia Ente
CONSORZI DI ENTI LOCALI DI CUI ALL' ART. 2 DEL TUEL                                                                               13
REGIONI E PROVINCE AUTONOME                                                                                                       10
ENTI STRUMENTALI IN CONTABILITA' FINANZIARIA INTEGRATA PER LO SVILUPPO SOSTENIBILE E LA TUTELA DEL TERRITORIO E DELL' AMBIENTE     9
Name: count, dtype: int64
Descrizione Tipologia Ente
CONSORZI DI ENTI LOCALI DI CUI ALL' ART. 2 DEL TUEL                                                                               214
REGIONI E PROVINCE AUTONOME                                                                                                       194
COMUNI                                                                                                                            190
CITTA' METROPOLITANE                                                                                                   

In [26]:
def is_numeric_column(series, threshold=0.8):
    """Returns True if at least 'threshold' percentage of non-nulls values that can be converted in a number"""
    non_null = series.replace('', pd.NA).dropna()
    if len(non_null) == 0:
        return False
    converted = pd.to_numeric(non_null.str.replace(',', '.'), errors='coerce')
    success_rate = converted.notna().sum() / len(non_null)
    return success_rate >= threshold

In [25]:
for df in exp_overview_mission_dfs.values():
    df.columns = [c.lower().strip().replace(' ', '_').replace("'", '') for c in df.columns]
    
    df.info()
    

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column                                                        Non-Null Count  Dtype
---  ------                                                        --------------  -----
 0   codice_voce_riepilogo_spese                                   32 non-null     str  
 1   esercizio_finanziario                                         32 non-null     str  
 2   ordine_esposizione_riepilogo_missioni_voce_di_riepilogo_ente  32 non-null     str  
 3   descrizione_zona                                              32 non-null     str  
 4   territorio_regione                                            32 non-null     str  
 5   territorio_provincia                                          32 non-null     str  
 6   territorio_comune                                             32 non-null     str  
 7   descrizione_ente                                              32 non-null     str  
 8   descrizione_t